# Arbitrage with Put-Call Parity

## Data

Use the data in `data/option_data_bb_NVDA.xlsx`
* tab `spot`: the underlying NVDA quote
* one tab per expiration date, (e.g. `2026-09-18`,): the option chain for that expiration

In [1]:
import sys
sys.path.insert(0, '../cmds')

import numpy as np
import pandas as pd
from pandas import IndexSlice
import matplotlib.pyplot as plt

from options import *

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 14

In [2]:
TICK = 'NVDA'
EXPRY = '2026-09-18'

FILEDATA = '../data/option_data_bb_NVDA.xlsx'

spot = pd.read_excel(FILEDATA, sheet_name='spot')
spot.rename(columns={'Unnamed: 0': 'field'}, inplace=True)
spot.set_index('field', inplace=True)

opt = pd.read_excel(FILEDATA, sheet_name=EXPRY)
opt.rename(columns={'Unnamed: 0': 'ticker'}, inplace=True)
opt.set_index('ticker', inplace=True)

In [3]:
styled = (
    spot.style
    .format("{:.2f}", subset=IndexSlice[['hist vol 30d', 'hist vol 60d', 'price'], :])
    .format("{:%Y-%m-%d}", subset=IndexSlice[['last update date'], :])
    .format("{:,.0f}", subset=IndexSlice[['volume'], :])
)

display(styled)

,NVDA US Equity
field,
name,NVIDIA Corp
last update date,2026-06-08
last update time,20:56:12.393006
price,208.64
volume,"138,372,837"
hist vol 30d,46.01
hist vol 60d,40.87


In [4]:
styled = (
    opt.iloc[1::8].style
    .format("{:.2f}", subset=IndexSlice[:, ['strike price', 'price', 'finance rate', 'implied vol', 'delta', 'gamma', 'vega', 'theta', 'rho', 'bid', 'ask', 'bid size', 'ask size']])
    .format(lambda x: x.strftime("%Y-%m-%d") if pd.notna(x) else "", subset=IndexSlice[:, ['last update date']], na_rep='')
    .format("{:,.0f}", subset=IndexSlice[:, ['volume', 'open int']])
)
display(styled)

,last update date,last update time,days to expiration,option type,strike price,price,finance rate,implied vol,delta,gamma,vega,theta,rho,bid,ask,bid size,ask size,open int,volume
ticker,,,,,,,,,,,,,,,,,,,
NVDA US 09/18/26 C135 Equity,2026-06-08,20:56:12.393006,102,Call,135.00,76.52,0.04,0.55,0.95,0.00,0.11,-0.02,0.00,75.55,77.15,66.00,42.00,"2,416",1
NVDA US 09/18/26 C175 Equity,2026-06-08,20:56:12.393006,102,Call,175.00,40.87,0.04,0.47,0.81,0.01,0.30,-0.06,0.00,41.35,41.95,33.00,40.00,"8,804",222
NVDA US 09/18/26 C215 Equity,2026-06-08,20:56:12.393006,102,Call,215.00,17.21,0.04,0.44,0.51,0.02,0.44,-0.09,0.00,17.40,17.60,62.00,22.00,"9,183",429
NVDA US 09/18/26 C255 Equity,2026-06-08,20:56:12.393006,102,Call,255.00,6.05,0.04,0.44,0.24,0.01,0.34,-0.07,0.00,6.10,6.25,102.00,42.00,"36,550",337
NVDA US 09/18/26 P140 Equity,2026-06-08,20:56:12.393006,102,Put,140.00,1.49,0.04,0.53,-0.05,0.00,0.12,-0.03,-0.00,1.47,1.51,185.00,155.00,"29,358",230
NVDA US 09/18/26 P180 Equity,2026-06-08,20:56:12.393006,102,Put,180.00,7.10,0.04,0.46,-0.22,0.01,0.33,-0.07,-0.00,7.10,7.30,112.00,73.00,"29,639","3,302"
NVDA US 09/18/26 P220 Equity,2026-06-08,20:56:12.393006,102,Put,220.00,25.05,0.04,0.43,-0.54,0.02,0.44,-0.09,-0.00,24.35,24.60,56.00,60.00,"8,335","1,097"
NVDA US 09/18/26 P260 Equity,2026-06-05,20:56:12.393006,102,Put,260.00,58.03,0.04,0.43,-0.81,0.01,0.30,-0.07,-0.00,54.10,55.35,33.00,14.00,221,19


#### A note on dividends

NVDA pays a small quarterly dividend (recently raised). Bloomberg's `finance rate` is an *implied* financing rate backed out from the option mids; for a dividend payer it reflects the **net** cost of carry (financing minus the dividend), so the put-call-parity check below already accounts for NVDA's dividend without a separate term. (Contrast this with the Black-Scholes exercise, whose AMZN options are on a stock that pays **no** dividend.)

# 1. Put-Call-Parity

Consider the option chain with expiration `2026-09-18`.

### 1.1.

For every listed strike, calculate how closely put-call parity holds. That is, compare the call spread $c-p$ to the present value moneyness, $S-K^*$.

Make a chart showing this error across strikes.

#### Note: Financing Rate

The financing rate is quoted from annual swaps with ACT/360 compounding. Thus, a quoted rate of $r$ and days-to-expiration of $\text{days}$ would be used to calculate the present value of some amount, $X$, as...

$$PV(X) = \frac{X}{1+\frac{\text{days}}{360} r}$$

### 1.2. Put-Call Arbitrage

Suppose you are going to put on a trade 
* sized to `1,000` long-short options contracts. 
* trading on the `ATM strike`, which is `210`.

(Recall that an option contract size is `100` shares, so the position will be `100,000` shares.)

Describe in detail the position you would take, including your positioning in the...
* calls
* puts
* shares
* cash

### 1.3.

What is the expected PnL of your trade? Detail what you expect to make in each leg of the trade.

### 1.4.

Is this an arbitrage? What risks are there?

Specify whether you are trading a **conversion** (short call, long put) or **reversal** (reverse conversion - long call, short put.)

### 1.5.

Suppose that the `ATM strike = 210` trade is not an arbitrage, but rather an indication about the true financing rate.

Instead of using the given financing rate, solve for a financing rate which sets the profits (at this strike) to 0, keeping put-call parity.

Recall that we're calculating a financing rate with ACT/360 annual compounding.

### 1.6.

Consider the strike at `270`. What is your expected profit from that arbitrage?

Aside from the financing rate, point to specific data which concerns you about putting on this trade.

# 2. Option Value

### 2.1.

For the `2026-09-18` expiration, create a plot of...
* Moneyness Ratio, $S/K$, on the x-axis
* call value on the y-axis
* put value on the y-axis

How would you describe the relationship? Does it look like the plots of the options' final payoffs?

### 2.2.

Let's now look across multiple expirations, at a single strike.

For the `ATM strike=210`, make a plot of...
* expiration date (or days to expiration) on the x-axis
* call value and put values on the y-axis

What do you notice in this relationship?

# 3. Building an Options Position

Use the `2026-09-18` market data.

### 3.1. 

For each of the following trades, calculate the cost of putting it on. 

### 3.2.

In a sentence, describe what exposure the position has. (How would it profit or lose?)

### 3.3. 

Plot the payoff of each of the three strategies.

#### Contract Size
Remember that a contract of calls or puts is for 100 units (shares)

| Package | Component | Position | Quantity | Strike |
|---------|-----------|----------|----------|--------|
| **<span style="color: midnightblue;">A</span>** | <span style="color: midnightblue;">Shares</span> | — | 0 | — |
|       | <span style="color: midnightblue;">Call</span> | **<span style="color: green;">Long</span>** | 1 | 210 |
|       | <span style="color: midnightblue;">Put</span> | **<span style="color: green;">Long</span>** | 1 | 210 |
|         |           |          |          |        |
| **<span style="color: steelblue;">B</span>** | <span style="color: steelblue;">Shares</span> | — | 0 | — |
|       | <span style="color: steelblue;">Call</span> | **<span style="color: red;">Short</span>** | 1 | 245 |
|       | <span style="color: steelblue;">Put</span> | **<span style="color: green;">Long</span>** | 1 | 195 |
|         |           |          |          |        |
| **<span style="color: cadetblue;">C</span>** | <span style="color: cadetblue;">Shares</span> | **<span style="color: green;">Long</span>** | 100 | — |
|       | <span style="color: cadetblue;">Call</span> | **<span style="color: red;">Short</span>** | 1 | 245 |
|       | <span style="color: cadetblue;">Put</span> | **<span style="color: green;">Long</span>** | 1 | 195 |